# Visualization of GPQR model

In [ ]:
import sys
import os
import warnings

import numpy as np
import torch
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))

warnings.filterwarnings("ignore")

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SLURRIES = ["G50", "G45", "G40", "G40+IPA"]

## 1D plot

In [ ]:
X = pd.read_csv("../../_temp/v1/X.csv", index_col=[0, 1, 2])
y = pd.read_csv("../../_temp/v1/y.csv", index_col=[0, 1, 2])
Xpred = pd.read_csv("../../_temp/v1/Xpred_1D.csv", index_col=[0, 1, 2])

In [ ]:
columns = ["slurry", "cosine_of_contact_angle"]
cos_thetas = X["cosine_of_contact_angle"].reset_index()[columns]
slurry_map = cos_thetas.drop_duplicates().set_index("cosine_of_contact_angle")["slurry"]

slurries = Xpred["cosine_of_contact_angle"].map(slurry_map)
unique_slurries = [s for s in SLURRIES if s in slurries.unique()]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

Rgt_pred = Xpred["gap_to_thickness_ratio"].unique()
Cas = Xpred["capillary_number"].unique()

Cas_sorted = np.sort(Cas)
cmap = plt.get_cmap("viridis", len(Cas_sorted))
norm = mcolors.BoundaryNorm(
    np.concatenate(
        [
            [Cas_sorted[0] * 0.9],
            (Cas_sorted[:-1] + Cas_sorted[1:]) / 2,
            [Cas_sorted[-1] * 1.1],
        ]
    ),
    ncolors=len(Cas_sorted),
)

In [ ]:
pred_gpqr = pd.read_csv("../../benchmarks/v1/gpqr.Xpred_1D.csv").pivot(
    index=["index", "batch", "quantile", "sample"],
    columns="target",
)

gpqr_levels = pred_gpqr.index.get_level_values("quantile").unique().sort_values()
mean_lowerq = (
    pred_gpqr["value"]
    .xs(gpqr_levels[0], level="quantile")
    .groupby(level="index")
    .mean()
)
mean_upperq = (
    pred_gpqr["value"]
    .xs(gpqr_levels[-1], level="quantile")
    .groupby(level="index")
    .mean()
)

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]
    this_y = y.loc[ok, "H"].to_numpy()

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred].copy()
    this_Xpred["prediction_index"] = np.flatnonzero(ok_pred)

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        prediction_index = this_Xpred.loc[ok_pred, "prediction_index"]
        ax.fill_between(
            this_Xpred.loc[ok_pred, "gap_to_thickness_ratio"],
            mean_lowerq.loc[prediction_index, "H"],
            mean_upperq.loc[prediction_index, "H"],
            facecolor=cmap(norm(ca)),
            edgecolor="none",
            alpha=0.3,
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("H")
fig.suptitle("Quantiles")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]
    this_y = y.loc[ok, "phi_1"].to_numpy()

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred].copy()
    this_Xpred["prediction_index"] = np.flatnonzero(ok_pred)

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        prediction_index = this_Xpred.loc[ok_pred, "prediction_index"]
        ax.fill_between(
            this_Xpred.loc[ok_pred, "gap_to_thickness_ratio"],
            mean_lowerq.loc[prediction_index, "phi_1"],
            mean_upperq.loc[prediction_index, "phi_1"],
            facecolor=cmap(norm(ca)),
            edgecolor="none",
            alpha=0.3,
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel(r"$\phi$")
fig.suptitle("Quantiles")
plt.show()